## TODO

### Schema, profile, and configuration

- [x] Separate the CSV input contract from workflow-managed fields: fix Provider and Rights to AGSL policy values, set Metadata Version from the global `Aardvark` setting, generate Modified once per conversion run in UTC, default Access Rights to `Public`, and default Suppressed to boolean `false`; allow record-level overrides only for Access Rights and Suppressed.
- [ ] Decide where workflow defaults and their override precedence live (for example, a versioned configuration file plus CLI options); reject unknown configuration keys and record the effective configuration in conversion diagnostics.
- [x] Map `Alternative Title` to `dct_alternative_sm`. Replace the AGSL-specific profile label `AGSL Call Number` with a separate optional `Call Number` input whose values are appended to `dct_identifier_sm` alongside the canonical ARK; update the profile, OpenIndexMaps fixture, template, and identifier tests without allowing the call number to replace the ARK.
- [ ] Compare every local profile obligation with current OGM documentation, document intentional AGSL overrides and their practical GeoBlacklight/Solr impact, and otherwise align the profile to the community standard.
- [ ] Resolve the `gbl_dateRange_drsim` contradiction between the profile/schema array type and OGM guidance showing one bracketed string; verify actual GeoBlacklight/Solr behavior and raise an OGM community issue if the published sources remain inconsistent.
- [x] Document that `dct_references_s` must be a JSON-encoded string inside the outer JSON document.

### Identifiers and output files

- [x] Validate `ID` before constructing filenames so missing values do not produce `nan_BL_Aardvark.json`.
- [x] Resolve and validate canonical `Identifier` and generated `ID` before constructing filenames so missing or conflicting values cannot produce invalid output filenames.
- [x] Replace the heuristic filename parsing with explicit ARK parsing to prevent filename collisions.
- [x] Preserve and test the critical ARK distinction: AGSL `id` uses the modified `ark:-77981-...` permalink form, while `dct_identifier_sm` uses normalized `ark:/77981/...`; never apply one normalization rule to both fields.
- [ ] Add optional NOID mint-and-bind support behind a testable service interface, preserving supplied IDs and making retries idempotent so the same source record cannot receive multiple ARKs.
- [ ] Define a parent/child identifier policy for sub-ARKs, including how canonical paths such as `ark:/77981/{parent}/{child}` map to collision-free GeoBlacklight `id` values, filenames, NOID bindings, and parent/child Aardvark relations.
- [x] Detect duplicate output filenames before files are overwritten.
- [x] Define current rerun behavior: overwrite JSON files with matching filenames, leave unrelated or stale files in the output directory untouched, and do not automatically clean the directory.
- [x] Remove the unused `index` variable or restore a deliberate index-based fallback.

### Spatial metadata and enrichment

- [x] Require a nonblank Geometry value for every input record, consistent with the current community JSON Schema.
- [ ] Validate supported `locn_geometry` syntax and coordinate ordering, and investigate deriving Geometry from Bounding Box or source extents using `ENVELOPE(W,E,N,S)`; report records with invalid or non-derivable geometry for review.
- [ ] Evaluate reusable bounding-box and controlled-value logic in `../Clean-Validate/04_clean-validate.ipynb`, but separate its raw `west,south,east,north` representation from Aardvark ENVELOPE ordering and replace silent corrections/defaults with logged, testable policy.
- [ ] Explore a reproducible Theme classification step using existing projects or controlled-vocabulary mappings; preserve human overrides and flag uncertain classifications rather than inventing them silently.

### Validation and tests

- [x] Report absent CSV input files, missing required CSV columns, malformed identifiers, and invalid numeric arrays with actionable errors; isolate row-level failures so remaining valid records can still be converted.
- [x] Validate the reference-URI and field-profile files before conversion, including file existence and expected column headers, and distinguish configuration failures from record-level metadata errors.
- [ ] Validate every generated record against `../../schema/geoblacklight-schema-aardvark.json` before final output and report all row/field errors together; update tests whenever the pinned community schema changes.
- [ ] Add representative conversion tests covering defaults and overrides, booleans, both ARK forms, Unicode, trimmed arrays, missing and minted IDs, references, geometry derivation, schema failures, blank rows, unknown columns, and decimal values.

### Parsing and normalization (Complete!)

- [x] Parse boolean fields explicitly instead of using `bool(value)`, which incorrectly treats strings such as `"false"` as `True`.
- [x] Only convert floats to integers when they represent whole numbers; avoid truncating legitimate decimal values.
- [x] Make `_im` array conversion tolerate whitespace and integer-like values such as `"1922.0"`.
- [x] Strip surrounding whitespace from every pipe-delimited value in Python, even when OpenRefine is used upstream, and test values such as `Index maps | Topographic Maps`.
- [x] Skip wholly blank CSV rows and test that they never create `nan_BL_Aardvark.json`; keep the blank template header-only.
- [x] Warn or fail on unknown input columns instead of silently ignoring them, including accidental columns such as `Column1`; decide explicitly whether `Github View` is removed or mapped to a supported/local reference URI.


## Step 1. Import Modules

In [1]:
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation
from pathlib import Path
import json

import pandas as pd

## Step 2. Specify the file paths

This workflow uses two different descriptions of Aardvark metadata:

- The **field profile** (`aardvark.csv`) maps human-readable CSV headings to Aardvark field names and types. The converter reads this file to decide how to transform each value.
- The **JSON Schema** (`../../schema/geoblacklight-schema-aardvark.json`) validates the finished JSON record. We will connect that separate validation step later.

Calling `aardvark.csv` a profile here keeps those two responsibilities distinct.

In [2]:
csv_file_path = Path('../csv2JSON/OpenIndexMaps_Aardvark_workshop.csv')  # the input CSV
reference_uris_file_path = Path('../aardvark-profile/referenceURIs.csv')  # reference URI labels and values
profile_file_path = Path('../aardvark-profile/aardvark.csv')  # friendly labels mapped to Aardvark fields and types
output_dir = Path('json_output')  # generated JSON records

### Define the AGSL workflow policy

Provider and Rights are fixed AGSL values.

Access Rights and Suppressed have defaults but may be overridden per record. 

Metadata Version is configurable for a conversion run and is expected to change infrequently.

These values are defined here before we connect them to record construction.

In [3]:
AGSL_PROVIDER = (
    "American Geographical Society Library – UWM Libraries"
)

AGSL_RIGHTS = [
    (
        '<a href="https://uwm.edu/libraries/digital-collections/'
        'copyright-digcoll/">Please see the UWM Libraries statement '
        "on Copyright and Digital Collections</a>."
    ),
    (
        "Although this data is being distributed by the American "
        "Geographical Society Library at the University of "
        "Wisconsin-Milwaukee Libraries, no warranty expressed or implied "
        "is made by the University as to the accuracy of the data and "
        "related materials. The act of distribution shall not constitute "
        "any such warranty, and no responsibility is assumed by the "
        "University in the use of this data, or related materials."
    ),
]

DEFAULT_ACCESS_RIGHTS = "Public"
ALLOWED_ACCESS_RIGHTS = {"Public", "Restricted"}

METADATA_VERSION = "Aardvark"
DEFAULT_SUPPRESSED = False

REQUIRED_REFERENCE_URI_COLUMNS = {
    "LABEL",
    "URI",
}

REQUIRED_PROFILE_COLUMNS = {
    "Label",
    "Field Name",
    "Field Type",
}

## Step 3. Define the input-loading functions

These functions define how the converter will load its three tabular inputs:

- the CSV records being converted;
- the reference URI mappings used to construct `dct_references_s`;
- the Aardvark field profile used to map friendly CSV labels to JSON fields.

In [4]:
def load_csv_records(csv_path):
    """Load source metadata records from a CSV file."""
    csv_path = Path(csv_path)

    if not csv_path.is_file():
        raise FileNotFoundError(
            f"CSV input file not found: {csv_path}"
        )

    return pd.read_csv(csv_path)


def load_reference_uris(reference_uris_path):
    """Load reference labels and return a label-to-URI mapping."""
    reference_uris_path = Path(reference_uris_path)

    if not reference_uris_path.is_file():
        raise FileNotFoundError(
            f"Reference URI file not found: {reference_uris_path}"
        )

    reference_uris_data = pd.read_csv(reference_uris_path)

    if set(reference_uris_data.columns) != {"LABEL", "URI"}:
        raise ValueError(
            "Reference URI file must contain exactly these columns: "
            "LABEL, URI."
        )

    return dict(
        zip(
            reference_uris_data["LABEL"],
            reference_uris_data["URI"],
        )
    )


def load_field_profile(profile_path):
    """Load the CSV profile that maps labels to Aardvark fields."""
    profile_path = Path(profile_path)

    if not profile_path.is_file():
        raise FileNotFoundError(
            f"Field profile file not found: {profile_path}"
        )

    profile_data = pd.read_csv(profile_path)

    if not REQUIRED_PROFILE_COLUMNS.issubset(profile_data.columns):
        raise ValueError(
            "Field profile must contain these columns: "
            "Label, Field Name, Field Type."
        )

    return profile_data

## Step 4. Define transformation helpers

These functions transform individual metadata values without reading files or writing output.

In [5]:
AGSL_NAAN = "77981"


def ark_to_agsl_id(identifier):
    """Convert a canonical AGSL ARK to the modified (url-safe) GeoBlacklight id form."""
    if not isinstance(identifier, str):
        raise TypeError("The canonical ARK must be a string.")

    canonical_ark = identifier.strip()
    expected_prefix = f"ark:/{AGSL_NAAN}/"

    if not canonical_ark.startswith(expected_prefix):
        raise ValueError(
            f"Expected an AGSL ARK beginning with {expected_prefix!r}; "
            f"received {canonical_ark!r}."
        )

    noid = canonical_ark.removeprefix(expected_prefix)

    if not noid:
        raise ValueError("The canonical ARK is missing its NOID name.")

    if "/" in noid:
        raise NotImplementedError(
            "Sub-ARK conversion is not implemented yet. "
            f"Received {canonical_ark!r}."
        )

    return f"ark:-{AGSL_NAAN}-{noid}"

In [6]:
def agsl_id_to_filename(agsl_id):
    """Build an Aardvark filename from a validated AGSL GeoBlacklight id."""
    if not isinstance(agsl_id, str):
        raise TypeError("The AGSL id must be a string.")

    agsl_id = agsl_id.strip()
    expected_prefix = f"ark:-{AGSL_NAAN}-"

    if not agsl_id.startswith(expected_prefix):
        raise ValueError(
            f"Expected an AGSL id beginning with {expected_prefix!r}; "
            f"received {agsl_id!r}."
        )

    noid = agsl_id.removeprefix(expected_prefix)

    if not noid:
        raise ValueError("The AGSL id is missing its NOID name.")

    return f"{noid}_BL_Aardvark.json"

In [7]:
def resolve_ark_fields(identifier, supplied_id=None):
    """Build and cross-check the two Aardvark identifier fields."""
    if pd.isna(identifier) or not str(identifier).strip():
        raise ValueError("Identifier must contain a canonical AGSL ARK.")

    canonical_identifier = str(identifier).strip()
    generated_id = ark_to_agsl_id(canonical_identifier)

    if pd.notna(supplied_id) and str(supplied_id).strip():
        supplied_id = str(supplied_id).strip()

        if supplied_id != generated_id:
            raise ValueError(
                "Identifier and ID do not represent the same ARK: "
                f"{canonical_identifier!r} generates {generated_id!r}, "
                f"but the CSV contains {supplied_id!r}."
            )

    return {
        "id": generated_id,
        "dct_identifier_sm": [canonical_identifier],
    }

In [8]:
def add_call_number(identifier_values, call_number):
    """Append an optional call number without replacing existing identifiers."""
    values = identifier_values.copy()

    if pd.notna(call_number) and str(call_number).strip():
        values.append(str(call_number).strip())

    return values

In [9]:
def parse_integer(value):
    """Parse an integer or integer-like value without truncating decimals."""
    text = str(value).strip()

    try:
        number = Decimal(text)
    except InvalidOperation as error:
        raise ValueError(f"Expected an integer; received {value!r}.") from error

    if not number.is_finite() or number != number.to_integral_value():
        raise ValueError(f"Expected a whole number; received {value!r}.")

    return int(number)

In [10]:
def split_multivalues(value, field_name):
    """Split a pipe-delimited CSV value into an Aardvark array."""
    values = [
        item.strip()
        for item in str(value).split("|")
    ]

    if field_name.endswith("_im"):
        return [parse_integer(item) for item in values]

    return values

In [11]:
def stringify_scalar(value):
    """Convert a scalar CSV value to text without truncating decimals."""
    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value)

In [12]:
def parse_boolean(value):
    """Convert common CSV Boolean representations to a JSON Boolean."""
    if isinstance(value, bool):
        return value

    text = str(value).strip().casefold()

    if text == "true":
        return True

    if text == "false":
        return False

    try:
        number = Decimal(text)
    except InvalidOperation as error:
        raise ValueError(
            f"Expected true, false, 1, or 0; received {value!r}."
        ) from error

    if number == 1:
        return True

    if number == 0:
        return False

    raise ValueError(
        f"Expected true, false, 1, or 0; received {value!r}."
    )

In [13]:
def resolve_access_rights(value):
    """Return a canonical Access Rights value, defaulting to Public."""
    if pd.isna(value) or not str(value).strip():
        return DEFAULT_ACCESS_RIGHTS

    supplied_value = str(value).strip()

    canonical_values = {
        allowed_value.casefold(): allowed_value
        for allowed_value in ALLOWED_ACCESS_RIGHTS
    }

    try:
        return canonical_values[supplied_value.casefold()]
    except KeyError as error:
        allowed = ", ".join(sorted(ALLOWED_ACCESS_RIGHTS))

        raise ValueError(
            f"Access Rights must be one of {allowed}; "
            f"received {supplied_value!r}."
        ) from error

In [14]:
def resolve_suppressed(value):
    """Return the Suppressed value, defaulting to false."""
    if pd.isna(value) or not str(value).strip():
        return DEFAULT_SUPPRESSED

    return parse_boolean(value)

#### Build Aardvark references

Aardvark requires `dct_references_s` to be a JSON-encoded string, even though it represents a mapping of reference URIs to URLs. The converter therefore builds a Python dictionary first and serializes it with `json.dumps()` when constructing the outer Aardvark record.

For example, the outer JSON contains:

```json
"dct_references_s": "{\"http://schema.org/url\": \"https://example.com/item\"}"

In [15]:
def build_references(row, reference_uri_dict):
    """Build the Aardvark reference dictionary for one CSV record."""
    references = {}

    for reference_label, reference_uri in reference_uri_dict.items():
        if pd.notna(row.get(reference_label)):
            references[reference_uri] = row[reference_label]

    return references

In [16]:
def current_utc_timestamp():
    """Return the current UTC timestamp in Aardvark format."""
    return (
        datetime.now(timezone.utc)
        .isoformat(timespec="seconds")
        .replace("+00:00", "Z")
    )

### Check the transformation helpers

These examples verify ARK conversion and identifier resolution, demonstrate rejection of unsupported or conflicting identifiers, and check multivalued field conversion.

In [17]:
assert (
    ark_to_agsl_id("ark:/77981/gmgs1j97737")
    == "ark:-77981-gmgs1j97737"
)

assert (
    ark_to_agsl_id("  ark:/77981/gmgs1j97737  ")
    == "ark:-77981-gmgs1j97737"
)

try:
    ark_to_agsl_id("ark:/12345/example")
except ValueError:
    pass
else:
    raise AssertionError("An ARK with a different NAAN should be rejected.")


try:
    ark_to_agsl_id(None)
except TypeError:
    pass
else:
    raise AssertionError("A non-string identifier should be rejected.")


try:
    ark_to_agsl_id("ark:/77981/")
except ValueError:
    pass
else:
    raise AssertionError("An ARK without a NOID name should be rejected.")


try:
    ark_to_agsl_id("ark:/77981/gmgs1j97737/child")
except NotImplementedError:
    pass
else:
    raise AssertionError("Sub-ARKs should be recognized as unsupported for now.")

ark_fields = resolve_ark_fields(
    identifier="ark:/77981/gmgs1j97737",
    supplied_id="ark:-77981-gmgs1j97737",
)

assert ark_fields == {
    "id": "ark:-77981-gmgs1j97737",
    "dct_identifier_sm": ["ark:/77981/gmgs1j97737"],
}

assert resolve_ark_fields(
    identifier="ark:/77981/gmgs1j97737"
) == ark_fields

try:
    resolve_ark_fields(
        identifier="ark:/77981/gmgs1j97737",
        supplied_id="ark:-77981-different",
    )
except ValueError:
    pass
else:
    raise AssertionError("Conflicting Identifier and ID values should fail.")

assert split_multivalues(
    "Maps|Datasets",
    "gbl_resourceClass_sm",
) == ["Maps", "Datasets"]

assert split_multivalues(
    "1922|1924|1927",
    "gbl_indexYear_im",
) == [1922, 1924, 1927]

assert split_multivalues(
    "eng",
    "dct_language_sm",
) == ["eng"]

assert split_multivalues(
    "Index maps | Topographic Maps",
    "gbl_resourceType_sm",
) == ["Index maps", "Topographic Maps"]

assert parse_integer("1922") == 1922
assert parse_integer(" 1922 ") == 1922
assert parse_integer("1922.0") == 1922

assert split_multivalues(
    "1922 | 1924.0 | 1927",
    "gbl_indexYear_im",
) == [1922, 1924, 1927]

try:
    parse_integer("1922.5")
except ValueError:
    pass
else:
    raise AssertionError("A decimal value must not be truncated to an integer.")

assert stringify_scalar(1922.0) == "1922"
assert stringify_scalar(1922.5) == "1922.5"
assert stringify_scalar("1922") == "1922"
assert stringify_scalar("Public") == "Public"

assert parse_boolean(True) is True
assert parse_boolean(False) is False
assert parse_boolean("TRUE") is True
assert parse_boolean("false") is False
assert parse_boolean(1) is True
assert parse_boolean(0) is False
assert parse_boolean("1.0") is True
assert parse_boolean("0.0") is False

try:
    parse_boolean("maybe")
except ValueError:
    pass
else:
    raise AssertionError("An unrecognized Boolean value should fail.")

geojson_row = pd.Series({
    "Format": "GeoJSON",
    "Download": "https://example.org/example.geojson",
    "Index Map": "https://example.org/example.geojson",
})

geojson_references = build_references(
    geojson_row,
    {
        "Download": "http://schema.org/downloadUrl",
        "GeoJSON": "http://geojson.org/geojson-spec.html",
        "Index Map": "https://openindexmaps.org",
    },
)

assert geojson_references == {
    "http://schema.org/downloadUrl": "https://example.org/example.geojson",
    "https://openindexmaps.org": "https://example.org/example.geojson",
}

assert (
    agsl_id_to_filename("ark:-77981-gmgs1j97737")
    == "gmgs1j97737_BL_Aardvark.json"
)

try:
    agsl_id_to_filename("unexpected-id")
except ValueError:
    pass
else:
    raise AssertionError("A malformed AGSL id should be rejected.")

assert resolve_access_rights(None) == "Public"
assert resolve_access_rights("") == "Public"
assert resolve_access_rights("   ") == "Public"
assert resolve_access_rights("Public") == "Public"
assert resolve_access_rights("restricted") == "Restricted"

try:
    resolve_access_rights("Private")
except ValueError:
    pass
else:
    raise AssertionError("Invalid Access Rights should be rejected.")

assert resolve_suppressed(None) is False
assert resolve_suppressed("") is False
assert resolve_suppressed("   ") is False
assert resolve_suppressed(False) is False
assert resolve_suppressed("false") is False
assert resolve_suppressed("true") is True
assert resolve_suppressed(1) is True

try:
    resolve_suppressed("maybe")
except ValueError:
    pass
else:
    raise AssertionError("Invalid Suppressed values should be rejected.")

timestamp = current_utc_timestamp()

assert timestamp.endswith("Z")
assert datetime.fromisoformat(
    timestamp.replace("Z", "+00:00")
).tzinfo is not None

canonical_identifiers = [
    "ark:/77981/gmgsvdncn9m"
]

identifiers_with_call_number = add_call_number(
    canonical_identifiers,
    "  050-b A-1:50,000 Series 1404  ",
)

assert identifiers_with_call_number == [
    "ark:/77981/gmgsvdncn9m",
    "050-b A-1:50,000 Series 1404",
]

# The original list must remain unchanged.
assert canonical_identifiers == [
    "ark:/77981/gmgsvdncn9m"
]

assert add_call_number(
    canonical_identifiers,
    None,
) == [
    "ark:/77981/gmgsvdncn9m"
]

assert add_call_number(
    canonical_identifiers,
    "   ",
) == [
    "ark:/77981/gmgsvdncn9m"
]

### Construct an Aardvark record

This function transforms one CSV row into an Aardvark record. It receives
the field profile and reference mappings explicitly, which keeps it
independent of notebook state and file-loading behavior.

In [18]:
def construct_json_data(
    row,
    ark_fields,
    profile_data,
    reference_uri_dict,
    modified_at
):
    """Construct one Aardvark record from a CSV row."""
    json_data = ark_fields.copy()

    json_data["gbl_mdVersion_s"] = METADATA_VERSION

    json_data["schema_provider_s"] = AGSL_PROVIDER

    json_data["dct_rights_sm"] = AGSL_RIGHTS.copy()

    json_data["gbl_mdModified_dt"] = modified_at

    json_data["dct_accessRights_s"] = resolve_access_rights(
        row.get("Access Rights")
    )

    json_data["gbl_suppressed_b"] = resolve_suppressed(
        row.get("Suppressed")
    )

    for _, profile_row in profile_data.iterrows():
        label = profile_row["Label"]
        field_name = profile_row["Field Name"]
        field_type = profile_row["Field Type"]

        # These fields have already been resolved
        if field_name in {
            "id",
            "dct_identifier_sm",
            "dct_accessRights_s",
            "gbl_suppressed_b",
            "gbl_mdVersion_s",
            "schema_provider_s",
            "dct_rights_sm",
            "gbl_mdModified_dt",
        }:
            continue

        if field_name == "dct_references_s":
            references = build_references(row, reference_uri_dict)

            if references:
                json_data[field_name] = json.dumps(references)

        elif pd.notna(row.get(label)):
            source_value = row.get(label)

            if field_type == "Array":
                json_data[field_name] = split_multivalues(
                    source_value,
                    field_name,
                )
            elif field_type == "Boolean or string":
                json_data[field_name] = parse_boolean(source_value)
            else:
                json_data[field_name] = stringify_scalar(source_value)

    return json_data

## Step 5. Define the input-validation helpers

These functions validate the structure and contents of the input CSV before
records are written:

- identify columns the converter does not recognize;
- identify required columns missing from the CSV;
- identify required values missing from an individual record.

In [19]:
REQUIRED_INPUT_COLUMNS = {
    "Identifier", # We use this to populate `id` and `dct_identifier_sm` in the Aardvark record.
    "Title", 
    "Resource Class",
    "Geometry",
}

def find_missing_columns(csv_data):
    """Return required input columns absent from the CSV."""
    return sorted(
        REQUIRED_INPUT_COLUMNS - set(csv_data.columns)
    )

def find_missing_required_values(row):
    """Return required input fields that are blank in one CSV row."""
    return sorted(
        column
        for column in REQUIRED_INPUT_COLUMNS
        if pd.isna(row.get(column))
        or not str(row.get(column)).strip()
    )

def find_unknown_columns(
    csv_data,
    profile_data,
    reference_uri_dict,
):
    """Return columns that are not used to construct Aardvark records."""
    accepted_columns = (
        set(profile_data["Label"])
        | set(reference_uri_dict)
    )

    return sorted(
        set(csv_data.columns) - accepted_columns
    )


## Step 6. Define the conversion workflow

This function coordinates the complete conversion process: loading the input
files, validating the CSV, constructing each Aardvark record, handling
record-level errors, preventing duplicate filenames, and writing valid JSON.

In [20]:
def prepare_record_for_output(
    row,
    profile_data,
    reference_uri_dict,
    modified_at,
):
    """Validate and transform one CSV row for output."""
    missing_values = find_missing_required_values(row)

    if missing_values:
        formatted_fields = ", ".join(missing_values)

        raise ValueError(
            f"Missing required values: {formatted_fields}."
        )

    ark_fields = resolve_ark_fields(
        identifier=row.get("Identifier"),
        supplied_id=row.get("ID"),
    )

    ark_fields["dct_identifier_sm"] = add_call_number(
        ark_fields["dct_identifier_sm"],
        row.get("Call Number"),
    )

    json_data = construct_json_data(
        row,
        ark_fields,
        profile_data,
        reference_uri_dict,
        modified_at,
    )

    file_name = agsl_id_to_filename(json_data["id"])

    return json_data, file_name

### Write one JSON record

This function serializes one completed Aardvark record to a specified path.
Keeping file writing separate makes record construction easier to test without
creating files.

In [21]:
def write_json_record(json_data, file_path):
    """Write one Aardvark record as formatted UTF-8 JSON."""
    file_path = Path(file_path)

    with file_path.open("w", encoding="utf-8") as json_file:
        json.dump(
            json_data,
            json_file,
            indent=4,
            ensure_ascii=False,
        )

In [22]:
def convert_csv_to_json(csv_file_path, reference_uris_file_path, profile_file_path, output_dir):
    output_dir = Path(output_dir)

    csv_data = load_csv_records(csv_file_path)
    reference_uri_dict = load_reference_uris(reference_uris_file_path)
    profile_data = load_field_profile(profile_file_path)

    modified_at = current_utc_timestamp()

    missing_columns = find_missing_columns(csv_data)

    if missing_columns:
        formatted_columns = ", ".join(missing_columns)

        raise ValueError(
            f"Missing required CSV columns: {formatted_columns}."
        )

    unknown_columns = find_unknown_columns(
        csv_data,
        profile_data,
        reference_uri_dict,
    )
    
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)

    # Iterate over each row in the CSV and generate JSON files
    conversion_errors = []
    records_written = 0
    generated_filenames = set()

    for csv_line_number, (_, row) in enumerate(
        csv_data.iterrows(),
        start=2,
    ):
        if row.isna().all():
            continue

        try:
            json_data, file_name = prepare_record_for_output(
                row,
                profile_data,
                reference_uri_dict,
                modified_at,
            )

            if file_name in generated_filenames:
                raise ValueError(
                    f"Duplicate output filename {file_name!r}."
                )

            generated_filenames.add(file_name)

        except (TypeError, ValueError, NotImplementedError) as error:
            conversion_errors.append(
                {
                    "csv_line_number": csv_line_number,
                    "error_type": type(error).__name__,
                    "message": str(error),
                }
            )
            continue

        file_path = output_dir / file_name
        
        # Write the JSON data to a file
        write_json_record(json_data, file_path)

        records_written += 1

    return {
        "records_written": records_written,
        "records_skipped": len(conversion_errors),
        "errors": conversion_errors,
        "unknown_columns": unknown_columns,
    }  

## Step 7. Run the conversion

In [23]:
try:
    conversion_result = convert_csv_to_json(
        csv_file_path,
        reference_uris_file_path,
        profile_file_path,
        output_dir,
    )
except (FileNotFoundError, ValueError) as error:
    print(f"Conversion failed: {error}")
else:
    print(
        f"Records written: "
        f"{conversion_result['records_written']}"
    )
    print(
        f"Records skipped: "
        f"{conversion_result['records_skipped']}"
    )

    for error in conversion_result["errors"]:
        print(
            f"CSV line {error['csv_line_number']} "
            f"({error['error_type']}): {error['message']}"
        )

    for column in conversion_result["unknown_columns"]:
        print(
            f"Warning: CSV column {column!r} "
            "was not included in the Aardvark records."
        )

    print(
        f"JSON files generated in directory: {output_dir}"
    )

Records written: 38
Records skipped: 0
JSON files generated in directory: json_output
